In [1]:
from dotenv import load_dotenv
import os
import time
import json
import requests
import pandas as pd

# Ruta absoluta o relativa al .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de Grok desde .env
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")

if CLAUDE_API_KEY:
    print("Clave de Claude cargada correctamente.")
else:
    raise ValueError("No se encontró la clave de Claude. Verifica la ruta del .env.")

Clave de Claude cargada correctamente.


In [2]:
def call_claude_api(prompt, text):
    """
    Sends a text and a prompt to Claude Sonnet 3.5 and returns the generated summary.
    """
    url = "https://api.anthropic.com/v1/messages"

    headers = {
        "x-api-key": CLAUDE_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }

    payload = {
        "model": "claude-sonnet-4-5",
        "max_tokens": 1024,
        "system": "You are an expert assistant specialized in simplifying biomedical language.",
        "messages": [
            {"role": "user", "content": f"{prompt}\n\nText:\n{text}"}
        ]
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        print(f"Status code: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            print("Response keys:", data.keys())
            output = data["content"][0]["text"]
            return output.strip(), elapsed
        else:
            print(f"HTTP error {response.status_code}: {response.text[:200]}")
            return None, elapsed

    except Exception as e:
        print("Request error:", e)
        return None, None

In [3]:
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [4]:
# Prompt para Claude
prompt = """Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words.

    Abstract of a biomedical study text:Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words."""

# Tomar las dos primeras filas del dataset
df_claude_test = df.head(2).copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Procesar las dos filas
for i, fila in df_claude_test.iterrows():
    print(f"\n Processing {fila['name']} ({i+1}/{len(df_claude_test)})...\n")
    resumen, tiempo = call_claude_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f"Response time: {tiempo:.2f} s\n")

# Agregar columna con los resúmenes generados
df_claude_test["gen_summary"] = gen_summaries

# Guardar en CSV
ruta_salida = "./prueba_2filasclaude2.csv"
df_claude_test.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Results saved to: {os.path.abspath(ruta_salida)}")

# Mostrar los resultados en pantalla
display(df_claude_test)


 Processing 10.1002-14651858.CD009781.pub2 (1/2)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
Response time: 20.87 s


 Processing 10.1002-14651858.CD010694.pub2 (2/2)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
Response time: 22.97 s

Results saved to: c:\Users\braya\OneDrive\Documentos\GitHub\Proyecto-PLN-FLAG\src\api_tests\prueba_2filasclaude2.csv


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Plain Language Summary: How Well Do Special ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,# Plain Language Summary\n\n---\n\n## Plain Ti...


In [5]:
# Código para procesar los 380 registros con Claude
# Prompt para Claude
prompt = """Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words.

    Abstract of a biomedical study text:Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words."""

# Copiamos el dataset completo
df_claude_final = df.copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Procesar todas las filas
for i, fila in df_claude_final.iterrows():
    print(f"\n Processing {fila['name']} ({i+1}/{len(df_claude_final)})...\n")
    resumen, tiempo = call_claude_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f"⏱️ Response time: {tiempo:.2f} s\n")

# Agregar la columna con los resúmenes generados
df_claude_final["gen_summary"] = gen_summaries

# Guardar en CSV
ruta_salida = "./results_claude2.csv"
df_claude_final.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Results saved to: {os.path.abspath(ruta_salida)}")

# Mostrar los primeros registros generados
display(df_claude_final.head(3))


 Processing 10.1002-14651858.CD009781.pub2 (1/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 24.07 s


 Processing 10.1002-14651858.CD010694.pub2 (2/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 22.86 s


 Processing 10.1002-14651858.CD009416.pub2 (3/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 23.33 s


 Processing 10.1002-14651858.CD004104.pub4 (4/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 23.32 s


 Processing 10.1002-14651858.CD012689.pub2 (5/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'sto

,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Plain Title**\n\nA Study About Eye Drops for...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Plain Language Summary**\n\n---\n\n**Plain T...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,# Plain Language Summary\n\n## Plain Title\n\n...


In [6]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_claude2.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  379 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Plain Title**\n\nA Study About Eye Drops for...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Plain Language Summary**\n\n---\n\n**Plain T...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,# Plain Language Summary\n\n## Plain Title\n\n...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,# Plain Language Summary\n\n## Plain Title\n\n...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,# Plain Language Summary\n\n## Plain Title\nDo...


In [7]:
# Filtrar la fila faltante
faltante = df_check[df_check["gen_summary"].isnull()]
print(f" Faltan {len(faltante)} resúmenes.")
display(faltante[["name", "article"]])

 Faltan 1 resúmenes.


,name,article
137,10.1002-14651858.CD009593.pub5,Background\r\nXpert MTB/RIF and Xpert MTB/RIF ...


In [8]:
# Reprocesar solo esa fila
if not faltante.empty:
    for i, fila in faltante.iterrows():
        print(f"\n Reprocesando {fila['name']}...\n")
        resumen, tiempo = call_claude_api(prompt, fila['article'])
        df_check.loc[i, "gen_summary"] = resumen if resumen else ""
        print(f"Reparado en {tiempo:.2f} s")

# Guardar nuevamente el CSV actualizado
df_check.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Archivo actualizado: {os.path.abspath(ruta_csv)}")


 Reprocesando 10.1002-14651858.CD009593.pub5...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
Reparado en 25.29 s
Archivo actualizado: c:\Users\braya\OneDrive\Documentos\GitHub\Proyecto-PLN-FLAG\src\api_tests\results_claude2.csv


In [9]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_claude.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Summary for General Audience:**\n\nCorneal a...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Summary for General Audience**\n\nVenous leg...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,## Plain Language Summary\n\n**Background**\nC...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,Summary for General Audience:\n\nWhen people w...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,## Plain Language Summary\n\n**Background**\n\...
